In [8]:
import gymnasium as gym
import math
import random
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gpu_utils

In [9]:
# --- CONFIGURATION ---
BATCH_SIZE = 64
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.01      # Allow it to get very good (only 1% random)
EPS_DECAY = 2000    # Decay slower, let it explore more
TAU = 0.001         # Update target network slower for stability
LR = 5e-4           # Slightly higher learning rate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gpu_utils.print_gpu_info()

GPU : NVIDIA GeForce RTX 3060
GPU capability: (8, 6)
GPU Memory : 11.62 GB
Multiprocessors: 28
Max Threads/MP : 1536


In [10]:
# --- 1. THE NETWORK ---
class DQN(nn.Module):
    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        # Slightly larger brain for a harder problem
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

In [11]:
# --- 2. REPLAY MEMORY ---
Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward'))

class ReplayMemory(object):
    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

In [15]:
# --- 3. TRAINING & DEMO ---
def main():
    print("🚀 preparing mission: LunarLander-v3")
    
    # Train "Blind" (No rendering) for speed
    env = gym.make("LunarLander-v3", render_mode=None)
    
    n_actions = env.action_space.n
    state, info = env.reset()
    n_observations = len(state)

    policy_net = DQN(n_observations, n_actions).to(device)
    target_net = DQN(n_observations, n_actions).to(device)
    target_net.load_state_dict(policy_net.state_dict())

    optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
    memory = ReplayMemory(50000) # Larger memory
    steps_done = 0

    def select_action(state):
        nonlocal steps_done
        sample = random.random()
        eps_threshold = EPS_END + (EPS_START - EPS_END) * \
            math.exp(-1. * steps_done / EPS_DECAY)
        steps_done += 1
        if sample > eps_threshold:
            with torch.no_grad():
                return policy_net(state).max(1).indices.view(1, 1)
        else:
            return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)

    def optimize_model():
        if len(memory) < BATCH_SIZE:
            return
        transitions = memory.sample(BATCH_SIZE)
        batch = Transition(*zip(*transitions))

        non_final_mask = torch.tensor(tuple(map(lambda s: s is not None, batch.next_state)), device=device, dtype=torch.bool)
        non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])
        state_batch = torch.cat(batch.state)
        action_batch = torch.cat(batch.action)
        reward_batch = torch.cat(batch.reward)

        state_action_values = policy_net(state_batch).gather(1, action_batch)
        next_state_values = torch.zeros(BATCH_SIZE, device=device)
        with torch.no_grad():
            next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values
        expected_state_action_values = (next_state_values * GAMMA) + reward_batch

        criterion = nn.SmoothL1Loss()
        loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
        optimizer.step()

    # --- TRAINING LOOP ---
    num_episodes = 600
    print(f"Starting Training for {num_episodes} episodes...")

    for i_episode in range(num_episodes):
        state, info = env.reset()
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        total_reward = 0
        
        for t in count():
            action = select_action(state)
            observation, reward, terminated, truncated, _ = env.step(action.item())
            reward = torch.tensor([reward], device=device)
            total_reward += reward.item()
            done = terminated or truncated

            if terminated:
                next_state = None
            else:
                next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

            memory.push(state, action, next_state, reward)
            state = next_state

            optimize_model()

            # Soft update
            target_net_state_dict = target_net.state_dict()
            policy_net_state_dict = policy_net.state_dict()
            for key in policy_net_state_dict:
                target_net_state_dict[key] = policy_net_state_dict[key]*TAU + target_net_state_dict[key]*(1-TAU)
            target_net.load_state_dict(target_net_state_dict)

            if done:
                # Print progress every 20 episodes
                if i_episode % 20 == 0:
                    print(f"Episode {i_episode} | Score: {total_reward:.2f}")
                break
    
    env.close()
    print("✅ Training Complete.")

    # --- DEMO LOOP (Watch it fly) ---
    print("🎥 Launching Demo Sequence (5 Flights)...")
    env = gym.make("LunarLander-v3", render_mode="human")
    
    for i in range(5):
        state, info = env.reset()
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        score = 0
        while True:
            # Greedy action (no random)
            with torch.no_grad():
                action = policy_net(state).max(1).indices.view(1, 1)
            
            observation, reward, terminated, truncated, _ = env.step(action.item())
            score += reward
            
            if terminated or truncated:
                print(f"Flight {i+1} Score: {score:.2f}")
                break
            
            state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)
            
    env.close()

In [16]:
if __name__ == '__main__':
    main()

🚀 preparing mission: LunarLander-v3


/home/saurabh-singh/miniconda3/envs/ml-gpu/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Starting Training for 600 episodes...
Episode 0 | Score: -112.96
Episode 20 | Score: 13.46
Episode 40 | Score: -119.94
Episode 60 | Score: -84.68
Episode 80 | Score: -43.08
Episode 100 | Score: 22.75
Episode 120 | Score: 268.64
Episode 140 | Score: 211.67


KeyboardInterrupt: 